[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/Pesquisa-Operacional-III-A/blob/main/02_Grafos_intro.ipynb)

# UNIVERSIDADE FEDERAL FLUMINENSE

**TEP00187 - PESQUISA OPERACIONAL III-A**

**Prof.: Diogo Ferreira de Lima Silva**

---
# AULA 02 — INTRODUÇÃO À TEORIA DOS GRAFOS COM NETWORKX
---

Grafos são a estrutura matemática por trás de boa parte dos problemas que veremos
na disciplina: caminho mais curto, árvore geradora mínima, fluxo máximo, fluxo de
custo mínimo, PERT/CPM. Antes de resolver esses problemas, precisamos aprender a
**representar** e **explorar** redes no computador.

## Objetivos da aula

Ao final desta aula você deverá ser capaz de:

1. Reconhecer o vocabulário básico de grafos (vértice, aresta, grau, caminho, ciclo, componente);
2. Construir grafos **não dirigidos**, **dirigidos**, **ponderados** e **multigrafos** com a biblioteca `networkx`;
3. Passar de uma representação para outra (lista de arestas, lista de adjacências, matriz de adjacências, matriz de incidência, `DataFrame`);
4. Consultar propriedades da rede (grau, densidade, conectividade, ciclos, subgrafos);
5. Desenhar redes com `matplotlib`, controlando *layout*, cores, tamanhos e rótulos;
6. Fazer uma primeira análise de importância de vértices (centralidade).

## Roteiro

| Seção | Assunto |
|---|---|
| 1 | Bibliotecas |
| 2 | O que é um grafo? |
| 3 | Construindo o primeiro grafo |
| 4 | Grau dos vértices e densidade |
| 5 | Formas de representar um grafo |
| 6 | Visualização |
| 7 | Grafos ponderados |
| 8 | Grafos dirigidos |
| 9 | Multigrafos |
| 10 | Caminhos, ciclos e conectividade |
| 11 | Subgrafos |
| 12 | Famílias clássicas de grafos |
| 13 | Do dado ao grafo: `pandas` |
| 14 | Uma primeira análise de rede: centralidade |
| 15 | Exercícios |

---
## 1. Bibliotecas
---

Usaremos três bibliotecas ao longo de todo o curso:

- **`networkx`**: criação, manipulação e análise de grafos/redes;
- **`matplotlib`**: visualizações;
- **`pandas`**: tabelas (`DataFrame`), muito úteis para organizar e ler os resultados.

> Se estiver no **Google Colab**, essas bibliotecas já vêm instaladas.
> Localmente, instale com `pip install networkx matplotlib pandas`.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

print("networkx  :", nx.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("pandas    :", pd.__version__)

---
## 2. O que é um grafo?
---

Um **grafo** é um par $G = (V, A)$, em que:

- $V$ é um conjunto de **vértices** (ou nós);
- $A$ é um conjunto de **arestas** (ou arcos/ligações), isto é, pares de vértices.

Chamamos de **ordem** do grafo o número de vértices, $|V|$, e de **tamanho** o número
de arestas, $|A|$.

**Variações importantes:**

| Tipo | Significado | Classe do NetworkX |
|---|---|---|
| Não dirigido | a aresta $\{i,j\}$ não tem sentido: ir de $i$ a $j$ é o mesmo que ir de $j$ a $i$ | `nx.Graph` |
| Dirigido | o arco $(i,j)$ tem sentido; $(i,j) \neq (j,i)$ | `nx.DiGraph` |
| Ponderado | cada aresta carrega um número (distância, custo, capacidade, tempo...) | qualquer classe, com atributo de peso |
| Multigrafo | admite mais de uma aresta entre o mesmo par de vértices | `nx.MultiGraph` / `nx.MultiDiGraph` |

**Por que isso importa em Pesquisa Operacional?** Alguns exemplos:

- vértices = cidades, arestas = rodovias, peso = distância → **caminho mais curto**;
- vértices = locais, arestas = trechos possíveis de dutos, peso = custo → **árvore geradora mínima**;
- vértices = pontos da rede, arcos = tubulações, peso = capacidade → **fluxo máximo**;
- vértices = eventos de um projeto, arcos = atividades, peso = duração → **PERT/CPM**.

---
## 3. Construindo o primeiro grafo
---

Vamos criar um grafo **não dirigido** com 5 vértices (`A`...`E`) e 6 arestas.

O fluxo básico no NetworkX é sempre o mesmo:

1. instanciar o grafo (`nx.Graph()`);
2. adicionar vértices;
3. adicionar arestas.

In [ ]:
# 1) Instanciando um grafo NÃO DIRIGIDO vazio
G = nx.Graph()

# 2) Adicionando vértices a partir de uma lista
G.add_nodes_from(["A", "B", "C", "D", "E"])

# 3) Adicionando arestas a partir de uma lista de pares
G.add_edges_from([("A", "B"),
                  ("A", "C"),
                  ("B", "C"),
                  ("B", "D"),
                  ("C", "E"),
                  ("D", "E")])

G

### 3.1 Outras formas de construir o mesmo grafo

O NetworkX é flexível: existem vários caminhos para chegar ao mesmo grafo.
Repare em um detalhe importante: **ao adicionar uma aresta com um vértice que
ainda não existe, o vértice é criado automaticamente**.

In [ ]:
# Forma (a): um de cada vez
G_a = nx.Graph()
G_a.add_node("A")
G_a.add_edge("A", "B")     # o vértice "B" é criado automaticamente!

# Forma (b): passando a lista de arestas direto no construtor
arestas = [("A", "B"), ("A", "C"), ("B", "C"), ("B", "D"), ("C", "E"), ("D", "E")]
G_b = nx.Graph(arestas)

# Forma (c): a partir de um dicionário de adjacências
G_c = nx.Graph({"A": ["B", "C"],
                "B": ["A", "C", "D"],
                "C": ["A", "B", "E"],
                "D": ["B", "E"],
                "E": ["C", "D"]})

print("G_b é igual a G?", nx.utils.graphs_equal(G, G_b))
print("G_c é igual a G?", nx.utils.graphs_equal(G, G_c))

### 3.2 Inspecionando o grafo

Todo grafo do NetworkX expõe as *views* `G.nodes` e `G.edges`, além de vários
métodos de consulta. Essas *views* são "janelas" para o grafo: elas podem ser
percorridas com `for` e convertidas em lista.

In [ ]:
print("Vértices .................:", list(G.nodes))
print("Arestas ..................:", list(G.edges))
print("Ordem   |V| ..............:", G.number_of_nodes())   # == len(G)
print("Tamanho |A| ..............:", G.number_of_edges())
print()
print("Vizinhos de B ............:", list(G.neighbors("B")))
print("Existe a aresta (A, D)? ..:", G.has_edge("A", "D"))
print("O vértice 'C' está em G? .:", "C" in G)
print("O grafo é dirigido? ......:", G.is_directed())

#### Vértices e arestas podem carregar **atributos**

Atributos são simplesmente dicionários associados a cada vértice/aresta. É assim
que guardamos, por exemplo, o nome de uma cidade, a distância de uma rodovia ou a
capacidade de um duto.

In [ ]:
H = G.copy()   # trabalhamos em uma cópia para não "sujar" o grafo G

# Atribuindo atributos aos vértices
H.nodes["A"]["tipo"] = "depósito"
H.nodes["E"]["tipo"] = "cliente"

# Atribuindo atributo a uma aresta
H.edges["A", "B"]["distancia"] = 12

# data=True devolve também o dicionário de atributos
print("Vértices com atributos:", list(H.nodes(data=True)))
print()
print("Arestas com atributos :", list(H.edges(data=True)))

**Percorrendo o grafo com `for`**

o padrão mais usado no dia a dia:

In [ ]:
for v in G.nodes:
    vizinhos = ", ".join(sorted(G.neighbors(v)))
    print(f"O vértice {v} é adjacente a: {vizinhos}")

---
## 4. Grau dos vértices e densidade
---

O **grau** de um vértice $v$, denotado $d(v)$, é o número de arestas incidentes a
ele. Em um grafo simples, o número de vizinhos.


In [ ]:
print("Graus:", dict(G.degree))

---
## 5. Formas de representar um grafo
---

O mesmo grafo pode ser guardado de várias maneiras. Cada representação é
conveniente para uma finalidade diferente, e o NetworkX converte entre elas com
uma linha de código.

### 5.1 Lista de adjacências

Para cada vértice, a lista de seus vizinhos. É compacta para grafos **esparsos**
(poucas arestas) e é a representação usada internamente pelo NetworkX.

In [ ]:
nx.to_dict_of_lists(G)

### 5.2 Matriz de adjacências

Matriz $n \times n$ em que a posição $(i,j)$ vale 1 se existe a aresta $\{i,j\}$
e 0 caso contrário. 

Em grafos não dirigidos ela é **simétrica**.

Ocupa $n^2$ posições, sendo cara para redes grandes e esparsas, mas muito prática para cálculos matriciais.

In [ ]:
print(nx.adjacency_matrix(G).todense())

In [ ]:
# Versão com rótulos, bem mais legível:
nx.to_pandas_adjacency(G, dtype=int)

> **Curiosidade útil:** elevando a matriz de adjacências à potência $k$, a
> posição $(i,j)$ passa a contar quantos **passeios** de exatamente $k$ arestas
> existem entre $i$ e $j$.

In [ ]:
A = nx.to_pandas_adjacency(G, dtype=int)
A2 = A @ A   # A ao quadrado
print("Número de passeios de comprimento 2 entre cada par de vértices:")
A2

### 5.3 Matriz de incidência

Matriz $|V| \times |A|$: a linha é o vértice, a coluna é a aresta, e o elemento
vale 1 se o vértice é uma das pontas daquela aresta. Cada coluna tem exatamente
dois valores 1 (as duas pontas da aresta).

In [ ]:
incidencia = pd.DataFrame(
    nx.incidence_matrix(G).todense().astype(int),
    index=list(G.nodes),
    columns=[f"{u}-{v}" for u, v in G.edges]
)
incidencia

---
## 6. Visualização
---

Desenhar a rede ajuda a entender o problema. 

**Atenção:** a posição dos vértices no desenho não tem significado matemático. Ela é escolhida por um algoritmo de *layout* apenas para deixar a figura legível.

A função mais direta é `nx.draw_networkx()`.

In [ ]:
plt.figure(figsize=(5, 4))
nx.draw_networkx(G, node_color="orange", node_size=700, font_weight="bold")
plt.title("Nosso primeiro grafo")
plt.axis("off")
plt.show()

### 6.1 Escolhendo o *layout*

Um *layout* é uma função que devolve um dicionário `{vértice: (x, y)}`. Guardar
essa posição em uma variável é **fundamental**: assim conseguimos desenhar o
grafo original e a solução de um problema **nas mesmas coordenadas**, permitindo
comparação visual (faremos muito isso nas próximas aulas).

Principais opções:

| Função | Ideia |
|---|---|
| `nx.spring_layout` | modelo de molas: arestas "puxam", vértices "se repelem" |
| `nx.circular_layout` | vértices igualmente espaçados em um círculo |
| `nx.kamada_kawai_layout` | tenta fazer a distância no desenho refletir a distância no grafo |
| `nx.shell_layout` | vértices organizados em camadas concêntricas |
| `nx.spectral_layout` | usa autovetores da matriz laplaciana |

> Dica: `spring_layout` é aleatório. Use `seed=` para obter **sempre o mesmo desenho**.

In [ ]:
layouts = {
    "spring_layout"      : nx.spring_layout(G, seed=42),
    "circular_layout"    : nx.circular_layout(G),
    "kamada_kawai_layout": nx.kamada_kawai_layout(G),
    "shell_layout"       : nx.shell_layout(G),
}

fig, eixos = plt.subplots(1, 4, figsize=(16, 4))

for eixo, (nome, pos) in zip(eixos, layouts.items()):
    nx.draw_networkx(G, pos=pos, ax=eixo,
                     node_color="lightsteelblue", node_size=600,
                     font_weight="bold")
    eixo.set_title(nome)
    eixo.axis("off")

plt.tight_layout()
plt.show()

### 6.2 Controle fino do desenho

`nx.draw_networkx()` é um atalho que chama, por baixo dos panos, três funções
separadas. Usá-las diretamente permite, por exemplo, **destacar** um subconjunto
de vértices ou arestas:

- `nx.draw_networkx_nodes()`
- `nx.draw_networkx_edges()`
- `nx.draw_networkx_labels()`

No exemplo abaixo destacamos os vértices de maior grau e o "caminho" A → C → E.

In [ ]:
pos = nx.spring_layout(G, seed=42)     # posição fixa, reutilizável

grau_maximo = max(dict(G.degree).values())
destaque    = [v for v, d in G.degree if d == grau_maximo]
demais      = [v for v in G.nodes if v not in destaque]

caminho_arestas = [("A", "C"), ("C", "E")]

plt.figure(figsize=(6, 5))

# arestas: primeiro todas em cinza, depois as destacadas por cima
nx.draw_networkx_edges(G, pos, edge_color="lightgray", width=2)
nx.draw_networkx_edges(G, pos, edgelist=caminho_arestas, edge_color="crimson", width=3)

# vértices
nx.draw_networkx_nodes(G, pos, nodelist=demais,   node_color="lightsteelblue", node_size=800)
nx.draw_networkx_nodes(G, pos, nodelist=destaque, node_color="crimson",        node_size=900)

# rótulos
nx.draw_networkx_labels(G, pos, font_weight="bold", font_color="white")

plt.title(f"Em vermelho: vértices de grau máximo ({grau_maximo}) e o caminho A-C-E")
plt.axis("off")
plt.show()

---
## 7. Grafos ponderados
---

Na prática, quase todas as arestas carregam um número: distância, custo, tempo,
capacidade, confiabilidade... Esse número é guardado como um **atributo da aresta**.

> **Convenção importante:** o NetworkX usa, por padrão, o atributo chamado
> **`weight`**. Se você nomear seu atributo `weight`, os algoritmos funcionam sem
> configuração extra. Se preferir outro nome (`distancia`, `custo`, `capacidade`),
> basta informá-lo no parâmetro `weight=` de cada algoritmo.

Vamos montar uma rede rodoviária simplificada do estado do Rio de Janeiro, com as
distâncias aproximadas em quilômetros.

In [ ]:
rodovias = [
    ("Rio de Janeiro", "Niterói",        15),
    ("Rio de Janeiro", "Petrópolis",     70),
    ("Niterói",        "Maricá",         40),
    ("Niterói",        "Itaboraí",       40),
    ("Maricá",         "Cabo Frio",     110),
    ("Itaboraí",       "Cabo Frio",     120),
    ("Itaboraí",       "Nova Friburgo", 100),
    ("Petrópolis",     "Nova Friburgo",  90),
    ("Cabo Frio",      "Macaé",          90),
    ("Nova Friburgo",  "Macaé",         130),
]

# add_weighted_edges_from grava o número no atributo 'weight'
RJ = nx.Graph()
RJ.add_weighted_edges_from(rodovias)

print(f"A rede tem {RJ.number_of_nodes()} cidades e {RJ.number_of_edges()} trechos.")

nx.to_pandas_edgelist(RJ, source="cidade A", target="cidade B")

### 7.1 Consultando os pesos

In [ ]:
# Peso de uma aresta específica
print("Rio de Janeiro -> Niterói:", RJ["Rio de Janeiro"]["Niterói"]["weight"], "km")

# Todos os pesos de uma vez, como dicionário
distancias = nx.get_edge_attributes(RJ, "weight")

# Os 3 trechos mais longos
print("\nTrechos mais longos:")
for (u, v), km in sorted(distancias.items(), key=lambda item: -item[1])[:3]:
    print(f"  {u:<15} - {v:<15} {km:>4} km")

# Extensão total da malha
print(f"\nExtensão total da malha: {RJ.size(weight='weight')} km")

> Repare na diferença: `RJ.size()` conta arestas; `RJ.size(weight="weight")`
> **soma os pesos**. O mesmo vale para `RJ.degree(weight="weight")`, que devolve
> o *grau ponderado* (soma dos pesos das arestas incidentes).

In [ ]:
pd.DataFrame(RJ.degree(weight="weight"),
             columns=["Cidade", "Km de rodovia incidente"]
            ).sort_values("Km de rodovia incidente", ascending=False)

### 7.2 Desenhando os pesos

Para mostrar os pesos no desenho usamos `nx.draw_networkx_edge_labels()`.

In [ ]:
pos_rj = nx.kamada_kawai_layout(RJ)

plt.figure(figsize=(20, 12))
nx.draw_networkx(RJ, pos_rj,
                 node_color="seagreen", node_size=2000,
                 font_size=8, font_color="white", font_weight="bold",
                 edge_color="gray")

nx.draw_networkx_edge_labels(RJ, pos_rj,
                             edge_labels=nx.get_edge_attributes(RJ, "weight"),
                             font_size=12, font_color="darkred")

plt.title("Rede rodoviária simplificada (distâncias em km)")
plt.axis("off")
plt.show()

### 7.3 Matriz de adjacências ponderada

Quando o grafo é ponderado, `to_pandas_adjacency` traz o peso no lugar do 1.
Pares sem aresta ficam com 0.

In [ ]:
nx.to_pandas_adjacency(RJ, weight="weight", dtype=int)

---
## 8. Grafos dirigidos
---

Em um **grafo dirigido** (`nx.DiGraph`) a aresta passa a se chamar **arco** e tem
sentido: $(i,j)$ significa "de $i$ para $j$" e é diferente de $(j,i)$.

Isso muda o conceito de grau, que se divide em dois:

- **grau de saída** ($d^+$): número de arcos que **saem** do vértice — `G.out_degree`;
- **grau de entrada** ($d^-$): número de arcos que **chegam** ao vértice — `G.in_degree`.

Exemplo: uma malha aérea com voos diretos (nem todo voo tem volta direta).

In [ ]:
voos = [
    ("GIG", "GRU"), ("GRU", "GIG"),
    ("GIG", "CNF"), ("CNF", "GIG"),
    ("GRU", "BSB"), ("BSB", "GRU"),
    ("CNF", "BSB"),
    ("BSB", "MAO"),
    ("MAO", "GRU"),
    ("GIG", "SDU"),
]

malha = nx.DiGraph()
malha.add_edges_from(voos)

print("É dirigido?", malha.is_directed())
print("Sucessores de GIG (destinos diretos):", list(malha.successors("GIG")))
print("Predecessores de GRU (origens diretas):", list(malha.predecessors("GRU")))

In [ ]:
tabela = pd.DataFrame({
    "Partidas (grau de saída)": dict(malha.out_degree),
    "Chegadas (grau de entrada)": dict(malha.in_degree),
})
tabela.index.name = "Aeroporto"
tabela.sort_values("Partidas (grau de saída)", ascending=False)

No desenho, arcos aparecem com **setas**. Quando existe voo nos dois sentidos,
o NetworkX desenha as duas setas em arcos ligeiramente curvos
(parâmetro `connectionstyle`).

In [ ]:
pos_malha = nx.circular_layout(malha)

plt.figure(figsize=(7, 6))
nx.draw_networkx(malha, pos_malha,
                 node_color="#20639B", node_size=1400,
                 font_color="white", font_weight="bold",
                 edge_color="gray", arrowsize=20,
                 connectionstyle="arc3,rad=0.12")
plt.title("Malha aérea (grafo dirigido)")
plt.axis("off")
plt.show()

### 8.1 Convertendo entre dirigido e não dirigido

Duas operações que usaremos bastante:

- `G.to_undirected()`: "esquece" o sentido dos arcos;
- `G.reverse()`: inverte o sentido de todos os arcos.

In [ ]:
nao_dirigido = malha.to_undirected()
invertido    = malha.reverse()

print("Arcos no grafo dirigido .........:", malha.number_of_edges())
print("Arestas após to_undirected() ....:", nao_dirigido.number_of_edges())
print("(pares com voo de ida e volta viram uma única aresta)")
print()
print("Destinos diretos a partir de GIG no grafo invertido:", list(invertido.successors("GIG")))

---
## 9. Multigrafos
---

Um **multigrafo** admite mais de uma aresta ligando o mesmo par de vértices —
útil quando existem, por exemplo, duas rodovias distintas entre as mesmas cidades,
ou vários voos com companhias e tarifas diferentes.

No NetworkX, cada aresta paralela recebe uma **chave** (`key`) que a identifica.

In [ ]:
multi = nx.MultiGraph()
multi.add_edge("Rio de Janeiro", "São Paulo", modal="avião",    tempo=1.0)
multi.add_edge("Rio de Janeiro", "São Paulo", modal="ônibus",   tempo=6.5)
multi.add_edge("Rio de Janeiro", "São Paulo", modal="carro",    tempo=5.5)
multi.add_edge("São Paulo",      "Campinas",  modal="ônibus",   tempo=1.5)

for u, v, chave, dados in multi.edges(keys=True, data=True):
    print(f"chave={chave} | {u} -> {v} | {dados['modal']:<7} | {dados['tempo']} h")

print("\nNúmero de ligações Rio-SP:", multi.number_of_edges("Rio de Janeiro", "São Paulo"))

> Na disciplina trabalharemos quase sempre com `nx.Graph` e `nx.DiGraph`.
> Quando houver arestas paralelas, normalmente ficamos apenas com a **melhor**
> delas (a mais barata, a mais rápida) e voltamos ao grafo simples.

---
## 10. Caminhos, ciclos e conectividade
---

Alguns conceitos centrais:

- **Passeio**: sequência de vértices em que cada par consecutivo é ligado por uma aresta;
- **Caminho**: passeio que não repete vértices;
- **Ciclo**: caminho que começa e termina no mesmo vértice;
- **Comprimento**: número de arestas do passeio (ou soma dos pesos, se ponderado);
- **Grafo conexo**: existe caminho entre **todo** par de vértices;
- **Componente conexa**: cada "pedaço" conexo máximo de um grafo desconexo.

### 10.1 Existe caminho? Qual o mais curto?

Nesta aula, "mais curto" significa **menos arestas** (grafo não ponderado).
A partir da Aula 04 trabalharemos com pesos e o algoritmo de Dijkstra.

In [ ]:
print("Existe caminho de Rio de Janeiro a Macaé?", nx.has_path(RJ, "Rio de Janeiro", "Macaé"))

rota = nx.shortest_path(RJ, "Rio de Janeiro", "Macaé")            # menos trechos
print("Rota com menos trechos :", " -> ".join(rota))
print("Número de trechos      :", nx.shortest_path_length(RJ, "Rio de Janeiro", "Macaé"))

# Note a diferença ao considerar as distâncias:
rota_km = nx.shortest_path(RJ, "Rio de Janeiro", "Macaé", weight="weight")
print("\nRota mais curta em km  :", " -> ".join(rota_km))
print("Distância percorrida   :", nx.shortest_path_length(RJ, "Rio de Janeiro", "Macaé", weight="weight"), "km")

E se quisermos **todos** os caminhos possíveis entre dois vértices?

In [ ]:
caminhos = list(nx.all_simple_paths(RJ, "Rio de Janeiro", "Macaé"))

print(f"Existem {len(caminhos)} caminhos simples entre Rio de Janeiro e Macaé:\n")
for c in sorted(caminhos, key=len):
    km = sum(RJ[c[i]][c[i + 1]]["weight"] for i in range(len(c) - 1))
    print(f"  {len(c) - 1} trechos | {km:>4} km | {' -> '.join(c)}")

### 10.2 Ciclos

`nx.cycle_basis()` devolve uma **base de ciclos**: um conjunto mínimo de ciclos a
partir do qual todos os demais podem ser obtidos. O número de ciclos dessa base é
$|A| - |V| + (\text{número de componentes})$ — o chamado *número ciclomático*.

In [ ]:
ciclos = nx.cycle_basis(RJ)

print("Base de ciclos da rede rodoviária:")
for c in ciclos:
    print("  ", " -> ".join(c), "-> ", c[0])

print(f"\nTamanho da base : {len(ciclos)}")
print(f"|A| - |V| + 1   : {RJ.number_of_edges() - RJ.number_of_nodes() + 1}")

### 10.3 Conectividade e componentes

Vamos montar propositalmente uma rede **desconexa** — uma rede de sensores em que
alguns grupos ficaram isolados — e identificar suas componentes.

In [ ]:
sensores = nx.Graph()
sensores.add_edges_from([
    ("s1", "s2"), ("s2", "s3"), ("s3", "s1"), ("s3", "s4"),   # componente 1
    ("s5", "s6"), ("s6", "s7"),                               # componente 2
    ("s8", "s9"),                                             # componente 3
])
sensores.add_node("s10")                                      # vértice isolado

print("A rede é conexa?", nx.is_connected(sensores))
print("Número de componentes:", nx.number_connected_components(sensores))
print()
for i, comp in enumerate(nx.connected_components(sensores), start=1):
    print(f"  Componente {i} ({len(comp)} sensores): {sorted(comp)}")

maior = max(nx.connected_components(sensores), key=len)
print("\nMaior componente:", sorted(maior))

Desenhando cada componente com uma cor diferente — repare como o `for` sobre
`connected_components` se combina naturalmente com `draw_networkx_nodes`.

In [ ]:
pos_s = nx.spring_layout(sensores, seed=7)
cores = ["#E63946", "#457B9D", "#2A9D8F", "#F4A261"]

plt.figure(figsize=(7, 5))
nx.draw_networkx_edges(sensores, pos_s, edge_color="gray")

for i, comp in enumerate(nx.connected_components(sensores)):
    nx.draw_networkx_nodes(sensores, pos_s, nodelist=list(comp),
                           node_color=cores[i % len(cores)], node_size=700,
                           label=f"Componente {i + 1}")

nx.draw_networkx_labels(sensores, pos_s, font_color="white", font_weight="bold")
plt.title("Rede de sensores desconexa: 4 componentes")
plt.legend(scatterpoints=1)
plt.axis("off")
plt.show()

### 10.4 Conectividade em grafos dirigidos

Em grafos dirigidos existem **duas** noções de conectividade:

- **fortemente conexo**: existe caminho respeitando o sentido dos arcos entre todo par de vértices;
- **fracamente conexo**: o grafo fica conexo se ignorarmos o sentido dos arcos.

In [ ]:
print("A malha aérea é fortemente conexa?", nx.is_strongly_connected(malha))
print("A malha aérea é fracamente conexa? ", nx.is_weakly_connected(malha))
print()
for comp in nx.strongly_connected_components(malha):
    print("  Componente fortemente conexa:", sorted(comp))

print("\nPor quê? Veja SDU:")
print("  chegadas:", malha.in_degree("SDU"), "| partidas:", malha.out_degree("SDU"))
print("  Uma vez em SDU, não há como sair — o grafo não é fortemente conexo.")

---
## 11. Subgrafos
---

Um **subgrafo** é um grafo formado por um subconjunto dos vértices e/ou das
arestas do grafo original. Vários problemas da disciplina são, no fundo,
"encontrar o melhor subgrafo com determinada propriedade" — a árvore geradora
mínima da próxima aula é exatamente isso.

- `G.subgraph(lista_de_vertices)`: mantém os vértices indicados e todas as arestas entre eles;
- `G.edge_subgraph(lista_de_arestas)`: mantém as arestas indicadas e seus extremos.

> As duas devolvem uma **visão somente leitura**. Use `.copy()` para obter um
> grafo independente que você possa modificar.

In [ ]:
litoral = RJ.subgraph(["Niterói", "Maricá", "Cabo Frio", "Macaé", "Itaboraí"]).copy()

print("Vértices:", list(litoral.nodes))
print("Arestas :", list(litoral.edges))
print("É conexo?", nx.is_connected(litoral))

fig, (e1, e2) = plt.subplots(1, 2, figsize=(13, 5))

nx.draw_networkx(RJ, pos_rj, ax=e1, node_color="lightgray", node_size=1200,
                 font_size=7, edge_color="lightgray")
nx.draw_networkx_nodes(RJ, pos_rj, ax=e1, nodelist=list(litoral.nodes),
                       node_color="darkorange", node_size=1200)
nx.draw_networkx_edges(RJ, pos_rj, ax=e1, edgelist=list(litoral.edges),
                       edge_color="darkorange", width=3)
e1.set_title("Subgrafo destacado dentro da rede completa")
e1.axis("off")

nx.draw_networkx(litoral, pos_rj, ax=e2, node_color="darkorange",
                 node_size=1200, font_size=7, font_weight="bold")
e2.set_title("O subgrafo isolado")
e2.axis("off")

plt.tight_layout()
plt.show()

### 11.1 Árvores: um subgrafo especial

Uma **árvore** é um grafo conexo e **sem ciclos**. Duas propriedades que usaremos
muito na próxima aula:

- toda árvore com $n$ vértices tem exatamente $n - 1$ arestas;
- existe **um único** caminho entre qualquer par de vértices de uma árvore.

In [ ]:
arvore = nx.Graph([("r", "a"), ("r", "b"), ("a", "c"), ("a", "d"), ("b", "e")])

print("É árvore?", nx.is_tree(arvore))
print(f"|V| = {arvore.number_of_nodes()}, |A| = {arvore.number_of_edges()} "
      f"(esperado |V| - 1 = {arvore.number_of_nodes() - 1})")
print("Tem ciclos?", len(nx.cycle_basis(arvore)) > 0)
print()
print("A rede rodoviária do RJ é uma árvore?", nx.is_tree(RJ), "-> ela tem ciclos.")

---
## 12. Famílias clássicas de grafos
---

O NetworkX traz dezenas de **geradores** prontos. Eles são ótimos para testar
algoritmos e para ilustrar conceitos.

In [ ]:
exemplos = {
    "complete_graph(6)\n(grafo completo)"   : nx.complete_graph(6),
    "cycle_graph(6)\n(ciclo)"               : nx.cycle_graph(6),
    "path_graph(6)\n(caminho)"              : nx.path_graph(6),
    "star_graph(5)\n(estrela)"              : nx.star_graph(5),
    "petersen_graph()\n(grafo de Petersen)" : nx.petersen_graph(),
    "gnp_random_graph(10, 0.3)\n(aleatório)": nx.gnp_random_graph(10, 0.3, seed=42),
}

fig, eixos = plt.subplots(2, 3, figsize=(14, 8))

for eixo, (nome, grafo) in zip(eixos.flatten(), exemplos.items()):
    nx.draw_networkx(grafo, pos=nx.spring_layout(grafo, seed=1), ax=eixo,
                     node_color="#6A4C93", node_size=400,
                     font_color="white", font_size=8, edge_color="gray")
    eixo.set_title(f"{nome}\n|V|={grafo.number_of_nodes()}, |A|={grafo.number_of_edges()}, "
                   f"densidade={nx.density(grafo):.2f}", fontsize=9)
    eixo.axis("off")

plt.tight_layout()
plt.show()

> **Grafo completo $K_n$:** todo par de vértices é ligado, logo
> $|A| = \binom{n}{2}$ e densidade 1. É o caso extremo — a maioria das redes
> reais é muito mais esparsa.

---
## 13. Do dado ao grafo: integrando com `pandas`
---

Na prática, os dados de uma rede chegam em uma planilha: uma linha por ligação.
O caminho `DataFrame` → grafo → `DataFrame` é usado o tempo todo.

In [ ]:
# Simulando a leitura de uma planilha (poderia ser pd.read_csv("arquivo.csv"))
dados = pd.DataFrame({
    "origem":   ["CD Duque de Caxias", "CD Duque de Caxias", "Loja Centro", "Loja Centro",
                 "Loja Barra", "Loja Niterói", "CD Duque de Caxias"],
    "destino":  ["Loja Centro", "Loja Barra", "Loja Barra", "Loja Niterói",
                 "Loja Niterói", "Loja São Gonçalo", "Loja São Gonçalo"],
    "custo":    [180, 240, 95, 130, 160, 85, 210],
    "tempo_min":[45, 60, 25, 35, 40, 20, 70],
})
dados

In [ ]:
# DataFrame -> grafo (podemos importar VÁRIOS atributos de uma vez)
D = nx.from_pandas_edgelist(dados, source="origem", target="destino",
                            edge_attr=["custo", "tempo_min"])

print("Vértices:", list(D.nodes))
print("Atributos da aresta CD -> Loja Centro:",
      D["CD Duque de Caxias"]["Loja Centro"])

In [ ]:
# Grafo -> DataFrame (de volta, agora já com os dados eventualmente modificados)
nx.to_pandas_edgelist(D)

Um detalhe prático: como o grafo tem **dois** atributos numéricos, precisamos
dizer a cada algoritmo qual deles usar através do parâmetro `weight=`.

In [ ]:
origem, destino = "CD Duque de Caxias", "Loja Niterói"

por_custo = nx.shortest_path(D, origem, destino, weight="custo")
por_tempo = nx.shortest_path(D, origem, destino, weight="tempo_min")

print("Minimizando CUSTO :", " -> ".join(por_custo),
      "| R$", nx.shortest_path_length(D, origem, destino, weight="custo"))
print("Minimizando TEMPO :", " -> ".join(por_tempo),
      "|", nx.shortest_path_length(D, origem, destino, weight="tempo_min"), "min")

---
---

## Para a próxima aula

Na **Aula 03** usaremos tudo isso para resolver o primeiro problema clássico de
otimização em redes: a **Árvore Geradora Mínima**: encontrar o subgrafo de menor
custo que mantém toda a rede conectada.

### Referências e documentação

- Documentação oficial do NetworkX: <https://networkx.org/documentation/stable/>
- Galeria de exemplos: <https://networkx.org/documentation/stable/auto_examples/index.html>
- Lista completa de geradores de grafos: <https://networkx.org/documentation/stable/reference/generators.html>
- Algoritmos disponíveis: <https://networkx.org/documentation/stable/reference/algorithms/index.html>